<h3>Estimating False Positives in SORD using Stratified Sampling Strategy</h3>

This notebook provides the python script used to extract samples for manual annotation of false positives present in SORD using a stratified sampling strategy considering the file types (QuestionsTitle, QuestionsBody, Answers, Comments), extraction method (Contain, LikeMinusContain) and the set of 19 keywords out of which 9 were selected as a sample. 

In [ ]:
import pyodbc
import pandas as pd

# --- CONNECTION ---
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=DESKTOP-EQTUGG1;"
    "DATABASE=SORecommendationsDb;"
    "Trusted_Connection=yes;"
)

# --- TABLES + TEXT COLUMNS ---
tables = {
    "QuestionsTitle_Contain": "Title",
    "QuestionsTitle_LikeMinusContain": "Title",
    "QuestionsBody_Contain": "Body",
    "QuestionsBody_LikeMinusContain": "Body",
    "Answers_Contain": "Body",
    "Answers_LikeMinusContain": "Body",
    "Comments_Like": "Text",
    "Comments_LikeMinusContain": "Text"
}

# --- KEYWORDS ---
keywords = ["Recommend", "Advocate", "Suggest", "Suitable", "Favor",
            "Oppose", "Condemn", "Champion", "Support"]

# --- ALLOCATION ---
allocation = {
    ("QuestionsTitle_Contain", "Suitable"): 1,
    ("QuestionsTitle_Contain", "Support"): 4,
    
    ("QuestionsTitle_LikeMinusContain", "Recommend"): 1,
    ("QuestionsTitle_LikeMinusContain", "Suggest"): 1,
    ("QuestionsTitle_LikeMinusContain", "Support"): 3,
    
    ("QuestionsBody_Contain", "Recommend"): 5,
    ("QuestionsBody_Contain", "Advocate"): 1,
    ("QuestionsBody_Contain", "Suggest"): 23,
    ("QuestionsBody_Contain", "Suitable"): 6,
    ("QuestionsBody_Contain", "Favor"): 1,
    ("QuestionsBody_Contain", "Condemn"): 1,
    ("QuestionsBody_Contain", "Champion"): 1,
    ("QuestionsBody_Contain", "Support"): 44,
    
    ("QuestionsBody_LikeMinusContain", "Recommend"): 10,
    ("QuestionsBody_LikeMinusContain", "Advocate"): 1,
    ("QuestionsBody_LikeMinusContain", "Suggest"): 55,
    ("QuestionsBody_LikeMinusContain", "Favor"): 4,
    ("QuestionsBody_LikeMinusContain", "Oppose"): 3,
    ("QuestionsBody_LikeMinusContain", "Condemn"): 2,
    ("QuestionsBody_LikeMinusContain", "Champion"): 3,
    ("QuestionsBody_LikeMinusContain", "Support"): 29,

    ("Answers_Contain", "Recommend"): 40,
    ("Answers_Contain", "Advocate"): 3,
    ("Answers_Contain", "Suggest"): 46,
    ("Answers_Contain", "Suitable"): 9,
    ("Answers_Contain", "Favor"): 3,
    ("Answers_Contain", "Oppose"): 1,
    ("Answers_Contain", "Condemn"): 1,
    ("Answers_Contain", "Champion"): 1,
    ("Answers_Contain", "Support"): 68,

    ("Answers_LikeMinusContain", "Recommend"): 15,
    ("Answers_LikeMinusContain", "Advocate"): 1,
    ("Answers_LikeMinusContain", "Suggest"): 34,
    ("Answers_LikeMinusContain", "Favor"): 5,
    ("Answers_LikeMinusContain", "Oppose"): 5,
    ("Answers_LikeMinusContain", "Condemn"): 2,
    ("Answers_LikeMinusContain", "Champion"): 2,
    ("Answers_LikeMinusContain", "Support"): 37,

    ("Comments_Like", "Recommend"): 25,
    ("Comments_Like", "Advocate"): 3,
    ("Comments_Like", "Suggest"): 46,
    ("Comments_Like", "Suitable"): 8,
    ("Comments_Like", "Favor"): 3,
    ("Comments_Like", "Condemn"): 2,
    ("Comments_Like", "Champion"): 1,
    ("Comments_Like", "Support"): 51,

    ("Comments_LikeMinusContain", "Recommend"): 14,
    ("Comments_LikeMinusContain", "Advocate"): 1,
    ("Comments_LikeMinusContain", "Suggest"): 85,
    ("Comments_LikeMinusContain", "Suitable"): 1,
    ("Comments_LikeMinusContain", "Favor"): 4,
    ("Comments_LikeMinusContain", "Oppose"): 6,
    ("Comments_LikeMinusContain", "Condemn"): 2,
    ("Comments_LikeMinusContain", "Champion"): 2,
    ("Comments_LikeMinusContain", "Support"): 34
}

# --- MAIN FUNCTION ---
def stratified_sample_to_csv(conn, tables, keywords, allocation):
    
    for table, text_col in tables.items():
        print(f"\nProcessing {table}...")
        table_samples = []

        for kw in keywords:
            n = allocation.get((table, kw), 0)
            if n == 0:
                continue

            query = f"""
                SELECT TOP ({n})
                       autoidpk,
                       id,
                       {text_col},
                       '{kw}' AS keyword
                FROM {table}
                WHERE {text_col} LIKE ?
                ORDER BY NEWID()
            """

            df = pd.read_sql(query, conn, params=[f"%{kw}%"])

            print(f"  {kw}: {len(df)} rows")

            if not df.empty:
                table_samples.append(df)

        # --- SAVE PER TABLE ---
        if table_samples:
            final_df = pd.concat(table_samples, ignore_index=True)

            # ensure clean types
            final_df["autoidpk"] = pd.to_numeric(final_df["autoidpk"], errors="coerce")
            final_df["autoidpk"] = final_df["autoidpk"].apply(lambda x: int(x) if pd.notnull(x) else None)

            for col in final_df.columns:
                if col != "autoidpk":
                    final_df[col] = final_df[col].apply(lambda x: str(x) if pd.notnull(x) else None)

            filename = f"stratified_{table}.csv"
            final_df.to_csv(filename, index=False)

            print(f"  ✅ Saved: {filename} ({len(final_df)} rows)")
        else:
            print(f"  ⚠️ No data for {table}")

# --- RUN ---
stratified_sample_to_csv(conn, tables, keywords, allocation)